## Packages:

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchinfo import summary
import argparse

## Data Loading:

In [2]:
class CIFAR10CustomDataset(Dataset):
    def __init__(self, csv_file=None, img_dir=None, transform=None, is_test=False, dataframe=None):
        """Load images from a CSV or a preselected pandas DataFrame.

        Args:
            csv_file (str, optional): Path to CSV annotations.
            img_dir (str): Directory containing images.
            transform (callable, optional): Image preprocessing/augmentation.
            is_test (bool): For unlabeled test data, return IDs instead of labels.
            dataframe (pd.DataFrame, optional): Selected rows for a train/validation split.
                Supply exactly one of csv_file and dataframe.
        """
        if (csv_file is None) == (dataframe is None):
            raise ValueError("Provide exactly one of csv_file or dataframe.")
        self.df = (pd.read_csv(csv_file) if dataframe is None
                   else dataframe.reset_index(drop=True).copy())
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        img_name = self.df.iloc[index]['filename']
        img_path = os.path.join(self.img_dir, img_name)
        with Image.open(img_path) as img:
            image = img.convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            img_id = self.df.iloc[index]['id']
            return image, img_id
        else:
            label = int(self.df.iloc[index]['label'])
            return image, torch.tensor(label, dtype=torch.long)

In [3]:
# --- 1. Transforms & Augmentations ---
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # Apply augmentation only to training images
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

# Deterministic preprocessing for validation and Kaggle test images.
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

# --- 2. Stratified 90/10 Training/Validation Split ---
labels_df = pd.read_csv('./data/train_labels.csv')
train_df, val_df = train_test_split(
    labels_df,
    test_size=0.1,
    random_state=42,
    stratify=labels_df['label'] # training and validation set get the same proportions of each class
)

train_dataset = CIFAR10CustomDataset(
    dataframe=train_df, 
    img_dir='./data/train_images',
    transform=train_transforms, 
    is_test=False
)
val_dataset = CIFAR10CustomDataset(
    dataframe=val_df, 
    img_dir='./data/train_images',
    transform=test_transforms, 
    is_test=False
)
test_dataset = CIFAR10CustomDataset(
    csv_file='./data/test.csv', 
    img_dir='./data/test_images',
    transform=test_transforms, 
    is_test=True
)

# --- 3. Wrap in DataLoaders ---
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f'Training: {len(train_dataset):,} | Validation: {len(val_dataset):,} | '
      f'Kaggle test: {len(test_dataset):,}')

Training: 45,000 | Validation: 5,000 | Kaggle test: 2,000


## Scaled Down ResNet18 Model

In [4]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out
    

In [5]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 40

        self.conv1 = nn.Conv2d(3, 40, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(40)
        self.layer1 = self._make_layer(block, 40, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 80, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 160, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 320, num_blocks[3], stride=2)
        self.linear = nn.Linear(320*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

In [6]:
def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

Instantiate Model and check the number of parameters:

In [7]:
model = ResNet18()

In [10]:
stats = summary(
    model,
    input_size=(1, 3, 32, 32),
    device="cpu",
    verbose=0
)

print(f"Total parameters: {stats.total_params:,}")
print(f"Trainable parameters: {stats.trainable_params:,}")

Total parameters: 4,368,690
Trainable parameters: 4,368,690
